In [62]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict, Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator

In [63]:
load_dotenv()

model = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

In [64]:
class eval_schema(BaseModel):
    feedback: str = Field(description="Detailed feedback for the essay.")
    score: float = Field(description="Score out of 10.", ge=0, le=10)

In [65]:
structured_model = model.with_structured_output(eval_schema)

In [66]:
essay = """# My Dreams

Every person has dreams and goals that give meaning and direction to life. Dreams inspire us to work hard, overcome difficulties, and become better versions of ourselves. I also have many dreams for my future. I want to become a successful, knowledgeable, and independent person who can make my family proud and contribute positively to society.

One of my biggest dreams is to build a successful career in the field of technology, especially Artificial Intelligence and Machine Learning. Technology is changing the world rapidly, and I want to be part of this change. I dream of becoming an expert in my field, creating useful AI systems, and working on projects that can solve real-world problems. I want to continue learning new technologies and improving my skills throughout my life.

Another important dream of mine is to become financially independent. I want to have a stable career that allows me to support myself and my family. I believe that financial independence brings confidence and freedom. However, I do not want success to be measured only by money. For me, true success means having knowledge, skills, good character, and the ability to help others.

I also dream of making my parents and family proud. They have supported me throughout my journey, and I want my achievements to be a source of happiness for them. I hope that through my hard work and dedication, I can give them a comfortable and peaceful life.

I know that dreams do not become reality without effort. There may be failures, challenges, and moments when I feel discouraged. Instead of giving up, I want to learn from my mistakes and keep moving forward. I believe that consistency, patience, discipline, and faith in oneself are the keys to achieving great goals.

In the future, I also want to use my knowledge to help other people. Whether through technology, education, or other forms of contribution, I want my work to have a positive impact. My dream is not simply to become successful; it is to become a person whose success has a meaningful purpose.

In conclusion, my dreams give me motivation to work hard and look forward to the future. I know that achieving them will take time and dedication, but I am ready to face the challenges along the way. I believe that with determination, continuous learning, and hard work, I can turn my dreams into reality.
"""

In [67]:
prompt = f'Evaluate the language quality of following essay and provide the feedback and assign a score out of 10 \n {essay}'
structured_model.invoke(prompt)


eval_schema(feedback='The essay is well‑organized and stays on topic throughout. The introduction clearly states the purpose, each paragraph develops a separate dream, and the conclusion ties the ideas together, giving the piece a logical flow. The language is generally clear and the vocabulary is appropriate for a personal essay; terms such as “financially independent,” “consistent,” and “discipline” convey the writer’s ideas effectively.\n\n**Strengths**\n- **Coherence and cohesion** – smooth transitions between paragraphs and a clear logical progression.\n- **Purposeful diction** – the writer uses concrete verbs ("build," "create," "support") that give the essay energy.\n- **Varied sentence length** – a mix of short and longer sentences keeps the rhythm engaging.\n\n**Areas for improvement**\n1. **Repetition** – the pronoun “I” begins many sentences, which can make the prose feel monotonous. Vary sentence openings (e.g., start with a clause, a gerund, or a rhetorical question).\n2. 

we are using 3 concepts in this workflow
1. Parallelism
2. structurre Output
3. Reducer Function

In [68]:
class State(TypedDict):
    essay: str
    clarity: str
    analysis: str
    language: str
    overall_feedback: str
    individual_score: Annotated[list[int], operator.add]
    avg_score: float

In [69]:
def eval_lang(state: State):

    essay = state["essay"]
    prompt = f'Evaluate the language quality of following essay and provide the feedback and assign a score out of 10 \n {essay}'
    structured_output = structured_model.invoke(prompt)
    return {"language": structured_output.feedback, "individual_score": [structured_output.score]}

In [70]:
def eval_clarity(state: State):

    essay = state["essay"]
    prompt = f'Evaluate the clarity of thought of following essay and provide the feedback and assign a score out of 10 \n {essay}'
    structured_output = structured_model.invoke(prompt)
    return {"clarity": structured_output.feedback, "individual_score": [structured_output.score]}

In [71]:
def eval_analysis(state: State):

    essay = state["essay"]
    prompt = f'Evaluate the depth of analysis of following essay and provide the feedback and assign a score out of 10 \n {essay}'
    structured_output = structured_model.invoke(prompt)
    return {"analysis": structured_output.feedback, "individual_score": [structured_output.score]}

In [72]:
def final_eval(state: State):

    prompt = f'Based on the following feedbacks create a sumarized feedback \n Language Feedback: {state["language"]} \n Clarity Feedback: {state["clarity"]} \n Analysis Feedback: {state["analysis"]} \n Also calculate the average score of the essay based on the individual scores provided in the list {state["individual_score"]}'
    model_output = model.invoke(prompt).content
    average_score = sum(state["individual_score"]) / len(state["individual_score"])
    return {"overall_feedback": model_output, "average_score": average_score}

In [73]:
graph = StateGraph(State)

# nodes
graph.add_node("eval_lang", eval_lang)
graph.add_node("eval_clarity", eval_clarity)
graph.add_node("eval_analysis", eval_analysis)
graph.add_node("final_eval", final_eval)

# edges
graph.add_edge(START, "eval_lang")
graph.add_edge(START, "eval_clarity")
graph.add_edge(START, "eval_analysis")
graph.add_edge("eval_lang", "final_eval")
graph.add_edge("eval_clarity", "final_eval")
graph.add_edge("eval_analysis", "final_eval")
graph.add_edge("final_eval", END)

workflow = graph.compile()


In [74]:
initial_state = {
    "essay": essay}

workflow.invoke(initial_state)

{'essay': '# My Dreams\n\nEvery person has dreams and goals that give meaning and direction to life. Dreams inspire us to work hard, overcome difficulties, and become better versions of ourselves. I also have many dreams for my future. I want to become a successful, knowledgeable, and independent person who can make my family proud and contribute positively to society.\n\nOne of my biggest dreams is to build a successful career in the field of technology, especially Artificial Intelligence and Machine Learning. Technology is changing the world rapidly, and I want to be part of this change. I dream of becoming an expert in my field, creating useful AI systems, and working on projects that can solve real-world problems. I want to continue learning new technologies and improving my skills throughout my life.\n\nAnother important dream of mine is to become financially independent. I want to have a stable career that allows me to support myself and my family. I believe that financial indepe